# 多层感知机的从零开始实现
实现一个多层感知机，与之前softmax回归获得的结果进行比较

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

## 初始化模型参数
因为 randn是N(0, 1)的标准正态分布，这里乘以 0.01 相当是N(0, 0.01^2)的正态分布

和 `torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)` 效果一致

In [ ]:
num_inputs, num_outputs, num_hiddens_1, num_hiddens_2 = 784, 10, 256, 128

W1 = nn.Parameter(torch.randn(
    num_inputs, num_hiddens_1, requires_grad=True) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens_1, requires_grad=True))
W2 = nn.Parameter(torch.randn(
    num_hiddens_1, num_hiddens_2, requires_grad=True) * 0.01)
b2 = nn.Parameter(torch.zeros(num_hiddens_2, requires_grad=True))
W3 = nn.Parameter(torch.randn(
    num_hiddens_2, num_outputs, requires_grad=True) * 0.01)
b3 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))

params = [W1, b1, W2, b2, W3, b3]

## 激活函数

In [ ]:
def relu(X):
    a = torch.zeros_like(X)
    return torch.max(X, a)

## 模型

In [ ]:
def net(X):
    X = X.reshape((-1, num_inputs))
    H1 = relu(X@W1 + b1)  # 这里“@”代表矩阵乘法
    H2 = relu(H1@W2 + b2)
    return (H2@W3 + b3)

## 损失函数

In [ ]:
loss = nn.CrossEntropyLoss(reduction='none')

## 训练

In [ ]:
def evaluate_accuracy(net, data_iter):  #@save
    """计算在指定数据集上模型的精度"""
    if isinstance(net, torch.nn.Module):
        net.eval()  # 将模型设置为评估模式
    metric = d2l.Accumulator(2)  # 正确预测数、预测总数
    with torch.no_grad():
        for X, y in data_iter:
            metric.add(d2l.accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]

def train_epoch_ch3(net, train_iter, loss, updater):  #@save
    """训练模型一个迭代周期（定义见第3章）"""
    # 将模型设置为训练模式
    if isinstance(net, torch.nn.Module):
        net.train()
    # 训练损失总和、训练准确度总和、样本数
    metric = d2l.Accumulator(3)
    for X, y in train_iter:
        # 计算梯度并更新参数
        y_hat = net(X)
        l = loss(y_hat, y)
        if isinstance(updater, torch.optim.Optimizer):
            # 使用PyTorch内置的优化器和损失函数
            updater.zero_grad()
            l.mean().backward()
            updater.step()
        else:
            # 使用定制的优化器和损失函数
            l.sum().backward()
            updater(X.shape[0])
        metric.add(float(l.sum()), d2l.accuracy(y_hat, y), y.numel())
    # 返回训练损失和训练精度
    return metric[0] / metric[2], metric[1] / metric[2]

def train_ch3(net, train_iter, test_iter, loss, num_epochs, updater):  #@save
    """训练模型（定义见第3章）"""
    animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs], ylim=[0.3, 0.9],
                        legend=['train loss', 'train acc', 'test acc'])
    for epoch in range(num_epochs):
        train_metrics = train_epoch_ch3(net, train_iter, loss, updater)
        test_acc = evaluate_accuracy(net, test_iter)
        animator.add(epoch + 1, train_metrics + (test_acc,))
    train_loss, train_acc = train_metrics
    assert train_loss < 0.5, train_loss
    assert train_acc <= 1 and train_acc > 0.7, train_acc
    assert test_acc <= 1 and test_acc > 0.7, test_acc

In [ ]:
num_epochs, lr = 20, 0.1
updater = torch.optim.SGD(params, lr=lr)
train_ch3(net, train_iter, test_iter, loss, num_epochs, updater)

# 简洁实现

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

## 模型

In [ ]:
# 暂退 dropout
dropout1, dropout2 = 0.2, 0.5
net = nn.Sequential(nn.Flatten(),
                    nn.Linear(784, 256),
                    nn.ReLU(),
                    nn.Dropout(dropout1),
                    nn.Linear(256, 128),
                    nn.ReLU(),
                    nn.Dropout(dropout2),
                    nn.Linear(128, 10))

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)

net.apply(init_weights);

## 训练

In [ ]:
batch_size, lr, num_epochs = 256, 0.1, 20
loss = nn.CrossEntropyLoss(reduction='none')
trainer = torch.optim.SGD(net.parameters(), lr=lr)
# 正则化
# wd = 0.01
# trainer = torch.optim.SGD([
#     {'params': [p for n, p in net.named_parameters() if 'weight' in n], 'weight_decay': wd},
#     {'params': [p for n, p in net.named_parameters() if 'bias' in n]}
# ], lr=lr)

train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)
train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)